<a href="https://colab.research.google.com/github/RDRamosU/mercado-laboral-steam-cr/blob/main/notebooks/03_limpieza_preparacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto 2 — Mercado Laboral Tech en Costa Rica
## Notebook 03 — Limpieza y preparación de datos

**Autor:** Ruben Dario Ramos Ulate
**Fecha:** Junio 2026  

---

## 1. Instalación e importación de librerías

In [4]:
!pip install pdfplumber openpyxl -q

import pandas as pd
import numpy as np
from google.colab import files

print("Librerías importadas ✓")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 94.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
Librerías importadas ✓


## 2. Carga del archivo ECE desde la computadora

In [5]:
# Subir archivo ECE directamente desde la computadora
print("Selecciona el archivo ECE...")
uploaded = files.upload()
for nombre in uploaded.keys():
    nombre_ece = nombre
    print(f"Archivo cargado: {nombre_ece} ✓")

Selecciona el archivo ECE...


Saving Principales+indicadores+EHPM+-+ECE+1987-2025+agrupado+Regiones+de+planficacin.xlsx to Principales+indicadores+EHPM+-+ECE+1987-2025+agrupado+Regiones+de+planficacin.xlsx
Archivo cargado: Principales+indicadores+EHPM+-+ECE+1987-2025+agrupado+Regiones+de+planficacin.xlsx ✓


In [6]:
# Reconstruir df_ece desde el archivo original
df_ece_raw = pd.read_excel(
    nombre_ece,
    sheet_name="Total",
    header=None,
    engine='openpyxl'
)

# Identificar columnas de años
fila_años = 4
year_row = df_ece_raw.iloc[fila_años]
año_a_col = {}
for col_idx, val in enumerate(year_row):
    if isinstance(val, (int, float)) and 2014 <= val <= 2025:
        año_a_col[int(val)] = col_idx

# Extraer indicadores
filas_indicadores = {
    'poblacion_total':           6,
    'fuerza_trabajo':           17,
    'ocupados':                 18,
    'desempleados':             19,
    'sector_primario':          27,
    'sector_secundario':        28,
    'sector_comercio_servicios':29,
    'ocup_calificada_alta':     33,
    'ocup_calificada_media':    34,
    'ocup_no_calificada':       35,
    'sector_publico':           43,
    'sector_privado':           44,
    'educacion_universitaria':  61,
    'tasa_participacion':       68,
    'tasa_ocupacion':           69,
    'tasa_desempleo':           71,
    'pct_subempleo':            78,
}

data = {'año': list(año_a_col.keys())}
for nombre, fila in filas_indicadores.items():
    data[nombre] = [df_ece_raw.iloc[fila, col] for col in año_a_col.values()]

df_ece = pd.DataFrame(data)

# Indicadores derivados
df_ece['pct_educacion_universitaria'] = (
    df_ece['educacion_universitaria'] / df_ece['ocupados'] * 100).round(1)
df_ece['pct_sector_servicios'] = (
    df_ece['sector_comercio_servicios'] / df_ece['ocupados'] * 100).round(1)
df_ece['pct_ocup_calificada_alta'] = (
    df_ece['ocup_calificada_alta'] / df_ece['ocupados'] * 100).round(1)

print(f"df_ece reconstruido ✓ — {df_ece.shape[0]} filas · {df_ece.shape[1]} columnas")

df_ece reconstruido ✓ — 12 filas · 21 columnas


## 3. Reconstrucción de datasets complementarios

In [7]:
# Dataset Radiografía Laboral CONARE 2019 y 2022
df_radiografia = pd.DataFrame({
    'disciplina': [
        'Ingeniería del Software',
        'Sistemas de Información',
        'Tecnologías de Información',
        'Ciencias de la Computación',
        'Ingeniería en Computadores',
        'Ingeniería Electrónica',
        'Ingeniería Industrial',
        'Ingeniería Civil',
        'Matemática',
        'STEM total',
        'No STEM total',
        'Nacional general'
    ],
    'clasificacion': [
        'STEM','STEM','STEM','STEM','STEM',
        'STEM','STEM','STEM','STEM',
        'STEM','No STEM','General'
    ],
    'desempleo_2019': [
        4.0, 4.6, 0.0, 1.1, 0.0,
        2.3, 3.8, 2.1, 3.5,
        2.5, None, 11.8
    ],
    'desempleo_2022': [
        None, None, None, None, None,
        None, None, None, None,
        4.2, 7.7, 12.2
    ],
    'subempleo_2019': [
        0.2, 0.7, 0.0, 0.0, 0.0,
        0.0, 0.0, 0.5, 1.2,
        None, None, None
    ],
    'trabajo_sin_relacion_2019': [
        4.4, 5.6, 2.2, 0.0, 4.2,
        4.2, 9.8, 7.3, 8.1,
        None, None, None
    ]
})

print(f"df_radiografia ✓ — {df_radiografia.shape[0]} filas · {df_radiografia.shape[1]} columnas")

df_radiografia ✓ — 12 filas · 6 columnas


In [8]:
# Dataset empleo tech — MICITT, BCCR, PROCOMER
df_empleo_tech = pd.DataFrame({
    'año': [2020, 2021, 2022, 2023, 2024],
    'empleo_zf_servicios':  [90829,  101676, 117310, 116242, 119982],
    'empleo_zf_total':      [144202, 164212, 184035, 186658, 197038],
    'empleo_empresas_tic':  [None,   None,   162737, None,   None],
    'empleo_tic_formal':    [42902,  58910,  None,   None,   None],
    'salario_zf_usd':       [1686,   1566,   1640,   2075,   2319],
    'salario_nacional_usd': [1181,   1155,   1121,   1157,   1169],
})

df_empleo_tech['brecha_salarial'] = (
    df_empleo_tech['salario_zf_usd'] /
    df_empleo_tech['salario_nacional_usd']
).round(2)

print(f"df_empleo_tech ✓ — {df_empleo_tech.shape[0]} filas · {df_empleo_tech.shape[1]} columnas")

df_empleo_tech ✓ — 5 filas · 8 columnas


In [9]:
# Dataset graduados STEAM — conecta con Proyecto 1
df_graduados = pd.DataFrame({
    'año': [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022],
    'steam_total':  [5169, 5525, 5711, 6468, 6154, 6654, 5830, 7007, 7273],
    'steam_mujer':  [2259, 2445, 2553, 2836, 2816, 3049, 2828, 3246, 3440],
    'steam_hombre': [2910, 3080, 3158, 3632, 3338, 3605, 3002, 3761, 3833],
    'fuente': 'OPES-CONARE'
})

print(f"df_graduados ✓ — {df_graduados.shape[0]} filas · {df_graduados.shape[1]} columnas")

df_graduados ✓ — 9 filas · 5 columnas


## 4. Verificación de calidad de datos

In [10]:
print("=== VERIFICACIÓN DE CALIDAD ===\n")

# 1. Valores nulos por dataset
for nombre, df in [('ECE', df_ece),
                   ('Radiografía', df_radiografia),
                   ('Empleo tech', df_empleo_tech),
                   ('Graduados', df_graduados)]:
    nulos = df.isnull().sum().sum()
    print(f"{nombre}: {nulos} valores nulos "
          f"({'esperados por diseño' if nulos > 0 else '✓ ninguno'})")

# 2. Rangos esperados
print("\n=== RANGOS DE INDICADORES CLAVE ===")
print(f"Tasa desempleo ECE: "
      f"{df_ece['tasa_desempleo'].min():.1f}% – "
      f"{df_ece['tasa_desempleo'].max():.1f}%")
print(f"Desempleo STEM 2022: 4.2% vs Nacional: 12.2%")
print(f"Brecha salarial ZF vs nacional: "
      f"{df_empleo_tech['brecha_salarial'].min():.2f}x – "
      f"{df_empleo_tech['brecha_salarial'].max():.2f}x")
print(f"Graduados STEAM: "
      f"{df_graduados['steam_total'].min():,} – "
      f"{df_graduados['steam_total'].max():,}")

# 3. Consistencia temporal
print("\n=== COBERTURA TEMPORAL ===")
print(f"ECE:          {df_ece['año'].min()} – {df_ece['año'].max()}")
print(f"Empleo tech:  {df_empleo_tech['año'].min()} – {df_empleo_tech['año'].max()}")
print(f"Graduados:    {df_graduados['año'].min()} – {df_graduados['año'].max()}")
print(f"Radiografía:  2019 y 2022 (cortes transversales)")

=== VERIFICACIÓN DE CALIDAD ===

ECE: 0 valores nulos (✓ ninguno)
Radiografía: 16 valores nulos (esperados por diseño)
Empleo tech: 7 valores nulos (esperados por diseño)
Graduados: 0 valores nulos (✓ ninguno)

=== RANGOS DE INDICADORES CLAVE ===
Tasa desempleo ECE: 6.7% – 19.6%
Desempleo STEM 2022: 4.2% vs Nacional: 12.2%
Brecha salarial ZF vs nacional: 1.36x – 1.98x
Graduados STEAM: 5,169 – 7,273

=== COBERTURA TEMPORAL ===
ECE:          2014 – 2025
Empleo tech:  2020 – 2024
Graduados:    2014 – 2022
Radiografía:  2019 y 2022 (cortes transversales)


## 5. Dataset integrado — periodo de solapamiento 2020–2022

Para comparar graduados STEAM vs empleo tech, usamos el periodo
donde todas las fuentes tienen datos: **2020–2022**.

In [11]:
# Merge ECE + Empleo tech para periodo 2020-2022
df_integrado = df_ece[df_ece['año'].isin([2020, 2021, 2022])][
    ['año', 'ocupados', 'tasa_desempleo',
     'educacion_universitaria', 'sector_comercio_servicios',
     'ocup_calificada_alta']
].merge(
    df_empleo_tech[['año', 'empleo_zf_servicios', 'empleo_zf_total',
                    'empleo_tic_formal', 'salario_zf_usd',
                    'salario_nacional_usd', 'brecha_salarial']],
    on='año'
).merge(
    df_graduados[df_graduados['año'].isin([2020, 2021, 2022])][
        ['año', 'steam_total', 'steam_mujer', 'steam_hombre']
    ],
    on='año'
)

# Calcular ratio empleo tech vs graduados STEAM
df_integrado['ratio_empleo_zf_vs_graduados'] = (
    df_integrado['empleo_zf_servicios'] /
    df_integrado['steam_total']
).round(1)

print("=== DATASET INTEGRADO 2020–2022 ===\n")
print(df_integrado[[
    'año', 'tasa_desempleo', 'steam_total',
    'empleo_zf_servicios', 'ratio_empleo_zf_vs_graduados',
    'brecha_salarial'
]].to_string(index=False))

print("\n📊 Interpretación del ratio:")
print("Por cada graduado STEAM, hay X empleos en ZF sector servicios tech")

=== DATASET INTEGRADO 2020–2022 ===

 año  tasa_desempleo  steam_total  empleo_zf_servicios  ratio_empleo_zf_vs_graduados  brecha_salarial
2020       19.606885         5830                90829                          15.6             1.43
2021       16.433372         7007               101676                          14.5             1.36
2022       12.218831         7273               117310                          16.1             1.46

📊 Interpretación del ratio:
Por cada graduado STEAM, hay X empleos en ZF sector servicios tech


## 6. Resumen del proceso de limpieza y preparación

| Verificación | Resultado |
|---|---|
| Consistencia ECE (ocupados + desempleados ≈ fuerza trabajo) | ✓ Aprobado |
| Valores nulos en ECE | Ninguno |
| Valores nulos en Radiografía, Empleo tech, Graduados | Esperados por diseño |
| Cobertura temporal suficiente para análisis | ✓ 2014–2025 |
| Dataset integrado 2020–2022 construido | ✓ 3 registros · 14 columnas |

**Hallazgo clave de esta etapa:**

Por cada graduado STEAM de universidades estatales, existen entre
**14.5 y 16.1 empleos** solo en el sector servicios tech de Zona Franca
(2020–2022). Sin contar empleo TIC formal ni empresas exportadoras.

Esto cuantifica con precisión la brecha de talento tech en Costa Rica.

In [12]:
# Exportar todos los datasets procesados
df_ece.to_csv("ece_indicadores_2014_2025.csv", index=False)
df_radiografia.to_csv("radiografia_laboral_stem_2019_2022.csv", index=False)
df_empleo_tech.to_csv("empleo_tech_2020_2024.csv", index=False)
df_graduados.to_csv("graduados_steam_2014_2022.csv", index=False)
df_integrado.to_csv("dataset_integrado_2020_2022.csv", index=False)

print("=== DATASETS EXPORTADOS ===")
print("  ece_indicadores_2014_2025.csv ✓")
print("  radiografia_laboral_stem_2019_2022.csv ✓")
print("  empleo_tech_2020_2024.csv ✓")
print("  graduados_steam_2014_2022.csv ✓")
print("  dataset_integrado_2020_2022.csv ✓")
print(f"\nListo para Notebook 04 — Visualizaciones y hallazgos")

=== DATASETS EXPORTADOS ===
  ece_indicadores_2014_2025.csv ✓
  radiografia_laboral_stem_2019_2022.csv ✓
  empleo_tech_2020_2024.csv ✓
  graduados_steam_2014_2022.csv ✓
  dataset_integrado_2020_2022.csv ✓

Listo para Notebook 04 — Visualizaciones y hallazgos


In [13]:
print("=" * 55)
print("RESUMEN — NOTEBOOK 03 COMPLETADO")
print("=" * 55)
print(f"\nDatasets procesados: 5")
print(f"  df_ece:          {df_ece.shape}")
print(f"  df_radiografia:  {df_radiografia.shape}")
print(f"  df_empleo_tech:  {df_empleo_tech.shape}")
print(f"  df_graduados:    {df_graduados.shape}")
print(f"  df_integrado:    {df_integrado.shape}")
print(f"\nHallazgo clave:")
print(f"  Ratio empleo ZF servicios vs graduados STEAM:")
for _, row in df_integrado.iterrows():
    print(f"  {int(row['año'])}: {row['ratio_empleo_zf_vs_graduados']:.1f}x")
print(f"\nSiguiente paso: Notebook 04 — Visualizaciones y hallazgos")

RESUMEN — NOTEBOOK 03 COMPLETADO

Datasets procesados: 5
  df_ece:          (12, 21)
  df_radiografia:  (12, 6)
  df_empleo_tech:  (5, 8)
  df_graduados:    (9, 5)
  df_integrado:    (3, 16)

Hallazgo clave:
  Ratio empleo ZF servicios vs graduados STEAM:
  2020: 15.6x
  2021: 14.5x
  2022: 16.1x

Siguiente paso: Notebook 04 — Visualizaciones y hallazgos
